In [ ]:
# lesson_01 > unit_03 > multi_turm.py

# from openai import OpenAI
# from dotenv import load_dotenv

# load_dotenv()
# client = OpenAI()

# def get_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         temperature=0.9,
#         messages=messages,
#     )
#     return response.choices[0].message.content

# messages = [ {"role": "system", "content": "너는 사용자를 도와주는 상담사야."} ]

# while True:
#     user_input = input("사용자: ")

#     if user_input == "exit":
#         break

#     messages.append({"role": "user", "content": user_input})
#     ai_response = get_response(messages)  # 대화 기록을 기반으로 AI 응답 가져오기
#     messages.append({"role": "assistant", "content": str(ai_response)})

#     print("AI: " + str(ai_response)) 

### 수동으로 메세지 추가

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv("../../.env")

model = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 금융 상담 역할
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        #SystemMessage("당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다.")
        ("placeholder", "{messages}"),  # 대화 이력 추가
        # MessagesPlaceholder("messages")
    ]
)

# 프롬프트와 모델을 연결하여 체인 생성
chain = prompt | model

# 이전 대화를 포함한 메시지 전달
ai_msg = chain.invoke(
    {
        "messages": [
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"),
            # HumanMessage("저축을 늘리기 위해 무엇을 할 수 있나요?"),
            ("assistant", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),  # 챗봇의 답변
            # AIMessage("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),
            ("human", "방금 뭐라고 했나요?"),  # 사용자의 재확인 질문
            # HumanMessage("방금 뭐라고 했나요?"),
        ],
    }
)
print(ai_msg.content)  # 챗봇의 응답 출력

저축을 늘리기 위해 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하는 방법을 추천했습니다. 이렇게 하면 저축 습관을 기르고, 필요한 자금을 쉽게 모을 수 있습니다. 추가적인 팁이나 질문이 있으시면 말씀해 주세요!


#### RunnableWithMessageHistory를 사용한 자동 대화 이력 관리

In [7]:
import sys

sys.path.append("C:/cloud_backend/langchain_app")

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from memory_history import get_session_history
from dotenv import load_dotenv

load_dotenv("../../.env")

model = ChatOpenAI(model="gpt-4o-mini")


with_message_history = RunnableWithMessageHistory(model, get_session_history)

c:\cloud_backend\langchain_app\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [2]:
config = {"configurable": {"session_id": "user01"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 김삿갓이야.")],
    config=config,
)

print(response.content)

안녕하세요, 김삿갓님! 반갑습니다. 어떻게 도와드릴까요?


In [3]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 김삿갓입니다. 맞나요? 혹시 다른 이름으로 불리고 싶으신가요?


In [4]:
config = {"configurable": {"session_id": "user02"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config
)

response.content

'죄송하지만, 당신의 이름을 알 수 없습니다. 제가 당신의 개인 정보를 저장하거나 기억할 수 없기 때문입니다. 대신 다른 질문이나 도움이 필요하시면 말씀해 주세요!'

In [5]:
config = {"configurable": {"session_id": "user01"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config
)

response.content

'우리는 당신의 이름이 김삿갓이라고 말씀하신 것에 대해 이야기했습니다. 그 외에 더 이야기하고 싶은 주제가 있으신가요?'

In [6]:
config = {"configurable": {"session_id": "user01"}}
response = with_message_history.invoke(
    [HumanMessage(content = "내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
)
print(response.content)

김삿갓이라는 이름은 한국의 전통적인 인물에서 유래한 것으로 보이므로, 한국 사람이라고 추측해봅니다. 

한국은 풍부한 문화 유산을 가지고 있으며, 다음과 같은 특징이 있습니다:

1. **한글**: 한국의 공식 문자로, 세종대왕에 의해 창제되었습니다. 간단하고 과학적인 구조로 인해 배우기 쉽습니다.

2. **음식**: 김치, 불고기, 비빔밥 등 다양한 전통 음식이 있으며, 최근에는 한국 음식이 세계적으로 인기를 끌고 있습니다.

3. **명절**: 설날(음력 1월 1일)과 추석(음력 8월 15일) 같은 전통 명절이 있으며, 가족과 함께 시간을 보내고 조상을 기립니다.

4. **예술**: 전통 음악인 국악과 전통 무용이 있으며, 현대 K-pop과 드라마, 영화가 세계적으로 인기를 얻고 있습니다.

5. **효**: 부모와 조상을 존중하는 효 사상이 중요하게 여겨집니다.

한국 문화에 대해 더 궁금한 점이 있으면 말씀해 주세요!
